# 05 - Spider Walking DMD

A short, self-contained example on a different animal: a walking spider. It runs the same PCA / FFT / DMD path as the hawk notebook (`02_bird_flight_dmd.ipynb`).

The spider skeleton is built into `morphing_birds`, so one packaged motion file is the only input. The last step animates a single DMD mode pair, showing the walking gait on its own.

> **Spider data** courtesy of Heiko Kabutz (CU Boulder) and Kaushik Jayaram (Imperial College London).

## Setup

Run this once in Colab to install the workshop package. Working locally from the repo? Skip the install line and run the imports straight away.

In [ ]:
# Run this once at the start of Colab.
!pip install -q uv
!uv pip install --system -q git+https://github.com/LydiaFrance/dmd-workshop.git@main

from animal_dmd import setup_workshop
setup_workshop();

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from birddmd import reconstruct, run_dmd
from IPython.display import display

import morphing_birds as mb
from morphing_birds import animate_plotly, animate_plotly_compare

from animal_dmd import (
    load_spider_motion,
    plot_marker_traces,
    plot_reconstruction_marker_traces,
    plot_unit_circle_eigenvalues,
    print_dmd_summary,
    plot_mode_pair_dynamics,
)

np.set_printoptions(precision=3, suppress=True)
warnings.filterwarnings(
    "ignore",
    message="Input data condition number.*",
    category=UserWarning,
)

## Load the Spider Walking Data

One packaged window of a walking spider. `load_spider_motion` returns the `Animal3D` skeleton for animation plus the motion array, times, marker names, and the frame interval `dt`. Times are in frames, so `dt = 1` and the DMD frequencies below are in cycles per frame.


In [ ]:
spider, markers, times, marker_names, dt = load_spider_motion()

print("Loaded the spider walking dataset")
print(f"{markers.shape[0]} frames, {markers.shape[1]} markers, {markers.shape[2]} coordinates")
print(f"{dt:.4f} frames between samples")

## Choose a Trace to Watch

DMD fits the whole spider at once, but the easiest sanity check is one coordinate. A leg-tip claw rises and falls once per step, so its vertical trace tracks the walking rhythm.

In [ ]:
selected_marker = "claw1"
selected_dimension = 2  # 0=x, 1=y, 2=z
dimension_labels = ["x", "y", "z"]

selected_marker_index = marker_names.index(selected_marker)
trace_label = f"{selected_marker} {dimension_labels[selected_dimension]}"

plot_marker_traces(
    times,
    {trace_label: markers[:, selected_marker_index, selected_dimension]},
    title=f"Input trace: {trace_label}",
)

## Animate the Raw Motion

Drag to rotate. The eight legs and the body motion should be visible before any DMD fitting.

In [ ]:
display(animate_plotly(spider, keypoints_frames=markers, axes_visible=True))

## Run DMD

Six modes with conjugate pairs enforced, so each oscillation in the walking gait comes out as one pair.

In [ ]:
mean_pose = markers.mean(axis=0, keepdims=True)

dmd_rank = 6
dmd_result = run_dmd(
    data=markers,
    times=times,
    n_modes=dmd_rank,
    d=2,
    eig_constraints={"conjugate_pairs"},
    average_shape=mean_pose,
    n_markers=markers.shape[1],
    verbose=False,
)

dmd_reconstruction = dmd_result.reconstruction
reconstruction_times = times[1:]
reconstruction_rmse = np.sqrt(np.mean((markers[1:] - dmd_reconstruction) ** 2))

growth_per_second = dmd_result.eigenvalues.real
frequency_hz = dmd_result.eigenvalues.imag / (2 * np.pi)

print_dmd_summary(
    growth_per_second,
    frequency_hz,
    reconstruction_rmse,
    label=f"rank-{dmd_rank} reconstruction",
)

## Check the Reconstruction

A quick check before reading the modes: does the rank-6 reconstruction follow the recorded claw trace? It begins at `times[1]`, since DMD only learns the step from one frame to the next.

In [ ]:
plot_reconstruction_marker_traces(
    reconstruction_times,
    {trace_label: markers[1:, selected_marker_index, selected_dimension]},
    {trace_label: dmd_reconstruction[:, selected_marker_index, selected_dimension]},
    title=f"DMD reconstruction check: {trace_label}",
)

In [ ]:
per_frame_eigenvalues = np.exp(dmd_result.eigenvalues * dt)
plot_unit_circle_eigenvalues(per_frame_eigenvalues)

## Compare Raw Motion and Reconstruction

Overlay the recorded walking motion (black) on the full DMD reconstruction (crimson). If they track each other, the chosen rank captures the motion well enough to interpret the individual mode pairs below.

In [ ]:
display(
    animate_plotly_compare(
        spider,
        keypoints_frames_list=[markers[1:], dmd_reconstruction],
        colours=["black", "crimson"],
    )
)

## Inspect DMD Mode Pairs

The walking gait appears as conjugate pairs. List the pair frequencies first, then build each pair's reconstruction so we can animate an individual mode pair on its own further down.

In [ ]:
pair_frequencies_hz = np.array(
    [dmd_result.pair_frequency(pair_index) for pair_index in range(dmd_result.n_pairs)]
)

for pair_index, pair_frequency in enumerate(pair_frequencies_hz):
    print(f"pair {pair_index}: {pair_frequency:.4f} cycles/frame")

In [ ]:
selected_pairs = list(range(dmd_result.n_pairs))

# Build each pair's reconstruction so we can animate a single mode pair below.
pair_reconstructions = {
    pair_index: reconstruct(dmd_result, pairs=[pair_index])
    for pair_index in selected_pairs
}

In [ ]:
# Each conjugate pair has one temporal dynamic of its own, built straight from
# the eigenvalue lambda = sigma + i*omega: a wave at frequency omega/(2*pi)
# inside an envelope e^{sigma t}. This is the mode's own rhythm in time, with no
# reference to any marker.
t0 = reconstruction_times - reconstruction_times[0]

mode_dynamics = {}
for pair_index in selected_pairs:
    mode_index = dmd_result.conjugate_pairs[pair_index][0]
    eigenvalue = dmd_result.eigenvalues[mode_index]   # sigma + i*omega
    growth = eigenvalue.real                           # < 0 decays, > 0 grows
    omega = eigenvalue.imag                            # angular frequency, rad/s

    mode_wave = np.exp(growth * t0) * np.cos(omega * t0)
    key = f"pair {pair_index}: {pair_frequencies_hz[pair_index]:.2f} cyc/frame, growth {growth:+.2f} /frame"
    mode_dynamics[key] = mode_wave

plot_mode_pair_dynamics(
    reconstruction_times, mode_dynamics, title="Mode-pair dynamics: frequency and growth/decay"
)
plt.show()


## Animate the Walking Mode

Animate one mode pair on the spider skeleton to see that part of the walking motion by itself. Change `pair_index` to inspect a different pair.

In [ ]:
mode_colours = ["blue", "green", "orange", "purple", "cyan"]
pair_index = 1

display(
    animate_plotly_compare(
        spider,
        keypoints_frames_list=[pair_reconstructions[pair_index]],
        colours=[mode_colours[pair_index % len(mode_colours)]],
    )
)

## What to Adjust Next

- Change `pair_index` in the animation to watch a different mode pair.
- Increase `dmd_rank` if the reconstruction misses obvious motion; decrease it if the modes look noisy.
- Switch `selected_marker` and `selected_dimension` to follow a different leg.
- Compare the pair frequencies with the spider's step rate to identify the gait mode.
